# Run Prioritized Cooperative A*
This notebook loads the grid and robot start/goal positions from the `results/` folder, runs the Cooperative A* planner, prints metrics, and visualizes the paths.

**Before running:** keep `cooperative_astar.py` in the same folder as this notebook and ensure `results/grid.csv`, `results/start.csv`, and `results/goal.csv` exist.

In [ ]:
import os
import time
import importlib

import matplotlib.pyplot as plt
import numpy as np

import cooperative_astar
importlib.reload(cooperative_astar)
from cooperative_astar import prioritized_cooperative_astar


## 1. Load the grid, starts, and goals

In [ ]:
RESULTS_DIR = "results"

grid_path = os.path.join(RESULTS_DIR, "grid.csv")
start_path = os.path.join(RESULTS_DIR, "start.csv")
goal_path = os.path.join(RESULTS_DIR, "goal.csv")

for path in (grid_path, start_path, goal_path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {path}. Run your grid generator first and make sure "
            "the results folder contains grid.csv, start.csv, and goal.csv."
        )

grid = np.loadtxt(grid_path, delimiter=",", dtype=int)
starts_array = np.atleast_2d(np.loadtxt(start_path, delimiter=",", dtype=int))
goals_array = np.atleast_2d(np.loadtxt(goal_path, delimiter=",", dtype=int))

starts = [tuple(map(int, row)) for row in starts_array]
goals = [tuple(map(int, row)) for row in goals_array]

print("Grid shape:", grid.shape)
print("Number of robots:", len(starts))
print("Starts:", starts)
print("Goals:", goals)


## 2. Run Cooperative A*

In [ ]:
start_time = time.perf_counter()
results = prioritized_cooperative_astar(grid, starts, goals)
elapsed = time.perf_counter() - start_time

print("Planning completed in", f"{elapsed:.6f}", "seconds")


## 3. Display per-robot results and summary metrics

In [ ]:
successful = 0
total_path_length = 0
total_nodes_expanded = 0
max_frontier = 0
makespan = 0

print("PRIORITIZED COOPERATIVE A* RESULTS")
print("=" * 45)

for i in range(len(starts)):
    result = results[f"Robot {i + 1}"]
    print(f"\nRobot {i + 1}:")
    print("  Start:", starts[i])
    print("  Goal:", goals[i])
    print("  Success:", result["success"])
    print("  Path length:", result["path_length"])
    print("  Arrival time:", result["arrival_time"])
    print("  Nodes expanded:", result["nodes_expanded"])
    print("  Maximum frontier size:", result["max_frontier_size"])

    total_nodes_expanded += result["nodes_expanded"]
    max_frontier = max(max_frontier, result["max_frontier_size"])
    if result["success"]:
        successful += 1
        total_path_length += result["path_length"]
        makespan = max(makespan, result["arrival_time"])

print("\nSUMMARY")
print("-" * 45)
print(f"Successful robots: {successful}/{len(starts)}")
print(f"Success rate: {successful / len(starts) * 100:.1f}%" if starts else "Success rate: N/A")
print(f"Total path length (successful robots): {total_path_length}")
print(f"Makespan: {makespan}")
print(f"Total nodes expanded: {total_nodes_expanded}")
print(f"Maximum frontier size observed: {max_frontier}")
print(f"Planning time: {elapsed:.6f} seconds")


## 4. Visualize all robot paths

In [ ]:
def visualize_paths(grid, starts, goals, results, title="Prioritized Cooperative A*"):
    rows, cols = grid.shape
    n = len(starts)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=1)
    colors = plt.cm.tab20(np.linspace(0, 1, max(n, 1)))

    for i in range(n):
        color = colors[i]
        result = results[f"Robot {i + 1}"]

        if result["success"]:
            path = result["path"]
            path_rows = [cell[0] for cell in path]
            path_cols = [cell[1] for cell in path]
            ax.plot(path_cols, path_rows, color=color, linewidth=2.2,
                    marker=".", markersize=5, label=f"Robot {i + 1} path", zorder=2)
        else:
            print(f"Robot {i + 1}: no conflict-free path found.")

        ax.scatter(starts[i][1], starts[i][0], color=color, s=180, marker="o",
                   edgecolors="black", linewidths=1.2,
                   label=f"Robot {i + 1} start", zorder=3)
        ax.scatter(goals[i][1], goals[i][0], color=color, s=230, marker="*",
                   edgecolors="black", linewidths=1.0,
                   label=f"Robot {i + 1} goal", zorder=3)

    ax.set_xticks(np.arange(-0.5, cols, 1))
    ax.set_yticks(np.arange(-0.5, rows, 1))
    ax.grid(color="gray", linewidth=0.5)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

visualize_paths(grid, starts, goals, results)
